# Markerless motion capture — you don't need the rig

Yesterday you used a **marker-based** system (OptiTrack): bright markers, a ring of cameras, a lab. Here you'll do motion capture with **an ordinary video and free software** — Google's **MediaPipe** finds your body joints in every frame, *no markers*.

**We'll walk through this together** — run each cell, and stop at the **Discuss** prompts. Then set the video source to **upload** and try it on **your own** movement (a squat, a reach, walking past the camera).

*Companion to the Day-1/Day-2 table-wiping notebooks.*

## Setup

In [ ]:
#@title Setup — install + load MediaPipe { display-mode: "form" }
# On Colab the installs run once (~30 s). Re-run this cell if you restart the runtime.
try:
    import google.colab; IN_COLAB = True
except Exception:
    IN_COLAB = False
import subprocess, sys, os, tempfile, urllib.request
try:
    import mediapipe, cv2
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe", "opencv-python-headless"], check=False)
import numpy as np, cv2, mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.core.base_options import BaseOptions
import matplotlib.pyplot as plt

MODEL = os.path.join(tempfile.gettempdir(), "pose_landmarker_full.task")
if not os.path.exists(MODEL):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/"
        "pose_landmarker_full/float16/latest/pose_landmarker_full.task", MODEL)

# BlazePose 33 landmarks: the skeleton edges (for drawing) and 3-point joints (for angles)
CONN = [(11,12),(11,23),(12,24),(23,24),(11,13),(13,15),(12,14),(14,16),
        (23,25),(25,27),(27,29),(24,26),(26,28),(28,30),(15,17),(16,18),(0,11),(0,12)]
JOINTS = {"right elbow":(12,14,16),"left elbow":(11,13,15),"right knee":(24,26,28),
          "left knee":(23,25,27),"right shoulder":(14,12,24),"left shoulder":(13,11,23)}

def run_pose(video):
    """Return per-frame landmarks (N,33,3 = x_px,y_px,visibility), the frames, fps, (W,H)."""
    cap = cv2.VideoCapture(video); fps = cap.get(cv2.CAP_PROP_FPS)
    W, H = int(cap.get(3)), int(cap.get(4))
    opts = vision.PoseLandmarkerOptions(base_options=BaseOptions(model_asset_path=MODEL),
            running_mode=vision.RunningMode.VIDEO, num_poses=1)
    LM, frames, i = [], [], 0
    with vision.PoseLandmarker.create_from_options(opts) as lm:
        while True:
            ok, fr = cap.read()
            if not ok: break
            res = lm.detect_for_video(
                mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)),
                int(i / fps * 1000))
            LM.append(np.array([[p.x*W, p.y*H, p.visibility] for p in res.pose_landmarks[0]])
                      if res.pose_landmarks else np.full((33,3), np.nan))
            frames.append(fr); i += 1
    cap.release()
    return np.array(LM), frames, fps, (W, H)

def draw(frame, pts):
    a = frame.copy()
    for u, v in CONN:
        if not (np.isnan(pts[u,0]) or np.isnan(pts[v,0])):
            cv2.line(a, tuple(pts[u,:2].astype(int)), tuple(pts[v,:2].astype(int)), (0,230,120), 2)
    for j in pts:
        if not np.isnan(j[0]): cv2.circle(a, tuple(j[:2].astype(int)), 3, (255,255,255), -1)
    return a

def show_video(path, width=480):
    """Embed a *playable* clip inline (re-encode to browser-friendly h264 first); fall back to a frame grid."""
    try:
        h264 = path.rsplit(".",1)[0] + "_h264.mp4"
        subprocess.run(["ffmpeg","-y","-i",path,"-vcodec","libx264","-pix_fmt","yuv420p",h264],
                       check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        from IPython.display import Video, display
        display(Video(h264, embed=True, width=width)); return
    except Exception:
        pass
    cap = cv2.VideoCapture(path); frames = []
    while True:
        ok, fr = cap.read()
        if not ok: break
        frames.append(fr)
    cap.release()
    if not frames: return
    idx = np.linspace(0, len(frames)-1, 6).astype(int)
    fig, ax = plt.subplots(2, 3, figsize=(12, 5))
    for k, ii in enumerate(idx):
        ax.flat[k].imshow(cv2.cvtColor(frames[ii], cv2.COLOR_BGR2RGB)); ax.flat[k].axis("off")
    plt.tight_layout(); plt.show()

def joint_angle(LM, joint):
    a, b, c = JOINTS[joint]
    ba = LM[:,a,:2]-LM[:,b,:2]; bc = LM[:,c,:2]-LM[:,b,:2]
    cos = (ba*bc).sum(-1)/(np.linalg.norm(ba,axis=-1)*np.linalg.norm(bc,axis=-1)+1e-9)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

def count_reps(sig, fps, min_sep_s=0.4):
    s = np.nan_to_num(sig - np.nanmean(sig))
    k = max(1, int(fps*0.1)); s = np.convolve(s, np.ones(k)/k, "same")
    thr = 0.3*np.nanstd(s); peaks, last = [], -1e9
    for i in range(1, len(s)-1):
        if s[i] > s[i-1] and s[i] >= s[i+1] and s[i] > thr and (i-last)/fps >= min_sep_s:
            peaks.append(i); last = i
    return peaks

SAMPLE_URL = "https://raw.githubusercontent.com/praneethnamburi/immersionlab-pe-mis/main/notebooks/assets/markerless_sample.mp4"
print("MediaPipe", mp.__version__, "ready.   IN_COLAB =", IN_COLAB)

## 1 · Get a video
Run as-is for the sample (a clip from the Day-1 capture), or set `source = "upload"` on Colab to use your own.

In [ ]:
#@title Get a video — upload your own, or use the sample { display-mode: "form" }
source = "sample"  #@param ["sample", "upload"]
if source == "upload" and IN_COLAB:
    from google.colab import files
    VIDEO = list(files.upload().keys())[0]
else:
    VIDEO = "assets/markerless_sample.mp4" if os.path.exists("assets/markerless_sample.mp4") \
            else os.path.join(tempfile.gettempdir(), "markerless_sample.mp4")
    if not os.path.exists(VIDEO):
        urllib.request.urlretrieve(SAMPLE_URL, VIDEO)
print("video:", VIDEO)

## 2 · Look at the raw data
Before any algorithm: this is what the camera saw. **Discuss —** looking at the raw video, what do you expect to be *easy* to track, and what will be *hard*? (lighting, occlusion, which body parts are visible, how fast things move)

In [ ]:
#@title Look at the raw data — play the video { display-mode: "form" }
# This is the raw input the algorithm sees: no tracking yet, just pixels.
show_video(VIDEO)

## 3 · Run markerless pose
MediaPipe finds 33 body landmarks in every frame — no markers, one camera. **Discuss —** where does the skeleton jitter, jump, or guess? What in the scene causes it?

In [ ]:
#@title Run markerless pose — the skeleton on your video { display-mode: "form" }
LM, frames, fps, (W, H) = run_pose(VIDEO)
tracked = float(np.mean(~np.isnan(LM[:,0,0]))*100)
print(f"{len(frames)} frames @ {fps:.0f} fps  ·  pose found in {tracked:.0f}% of frames")
annotated = os.path.join(tempfile.gettempdir(), "annotated.mp4")
vw = cv2.VideoWriter(annotated, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
for fr, pts in zip(frames, LM): vw.write(draw(fr, pts))
vw.release()
show_video(annotated)

## 4 · From skeleton to a number
A skeleton is only useful once it becomes a *measurement*. Pick a joint; we compute its angle over time and count reps/cycles. **Discuss —** is the rep count right? When would this simple counter fail?

In [ ]:
#@title From skeleton to a number — a joint angle + rep count { display-mode: "form" }
joint = "right elbow"  #@param ["right elbow","left elbow","right knee","left knee","right shoulder","left shoulder"]
ang = joint_angle(LM, joint)
t = np.arange(len(ang))/fps
reps = count_reps(ang, fps)
plt.figure(figsize=(9, 3.2))
plt.plot(t, ang, color="#0b7a52", lw=2)
plt.plot(t[reps], ang[reps], "o", color="#c026d3", label=f"{len(reps)} reps / cycles")
plt.xlabel("time (s)"); plt.ylabel(f"{joint} angle (deg)")
plt.title(f"{joint}: {np.nanmin(ang):.0f}-{np.nanmax(ang):.0f} deg  ·  {len(reps)} reps/cycles")
plt.legend(); plt.grid(alpha=.2); plt.tight_layout(); plt.show()
print(f"{joint}: range {np.nanmin(ang):.0f}-{np.nanmax(ang):.0f} deg;  {len(reps)} reps/cycles")

## 5 · How good is it? — vs. the ground truth
Your own clip has no "right answer" to check against. So we use the **Day-1 table-wiping**, captured with *both* a camera and the OptiTrack **marker rig**: run MediaPipe on the camera view and overlay its wrist against the rig's wrist. **Discuss —** the phone vs. the \$50k rig, on the same motion — how close?

In [ ]:
#@title How good is it? — markerless vs. the OptiTrack rig (Day-1 data) { display-mode: "form" }
# The catch with your own clip: there's no "right answer" to check against. So here we use the
# Day-1 table-wiping capture, which was recorded with BOTH a camera and the OptiTrack marker rig.
# We run MediaPipe on the camera view and overlay its wrist against the marker rig's wrist.
import h5py
try:
    from huggingface_hub import snapshot_download
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","-q","huggingface_hub"], check=False)
    from huggingface_hub import snapshot_download

CLIP = "assets/markerless_sample.mp4" if os.path.exists("assets/markerless_sample.mp4") \
       else os.path.join(tempfile.gettempdir(), "markerless_sample.mp4")
if not os.path.exists(CLIP): urllib.request.urlretrieve(SAMPLE_URL, CLIP)
H5 = os.path.join(snapshot_download("praneethnamburimit/immersionlab-pe-mis-table-wiping", repo_type="dataset"),
                  "table_wiping.h5")

LMc, _, fpsc, _ = run_pose(CLIP)          # markerless wrist (right wrist = landmark 16)
mpx = LMc[:, 16, 0]; tv = np.arange(len(mpx))/fpsc
with h5py.File(H5, "r") as f:             # marker-rig wrist over the "Normal" segment
    tot = f["mocap/t_ot"][:]
    wr = 0.5*(f["mocap/markers/ref_02_RWristThumb"][:] + f["mocap/markers/ref_03_RWristPinky"][:])
    n0, n1 = f["segments"].attrs["Normal"]
m = (tot >= n0) & (tot <= n1); tS = tot[m]-tot[m][0]; ot = wr[m][:, int(np.nanargmax(wr[m].var(0)))]
z = lambda a: (a-np.nanmean(a))/np.nanstd(a)
zmp = z(mpx); zot = z(np.interp(tv, tS, ot))
if np.corrcoef(zmp, zot)[0,1] < 0: zot = -zot
def _c(L): a,b = (zmp[L:],zot[:len(zot)-L]) if L>=0 else (zmp[:len(zmp)+L],zot[-L:]); return np.corrcoef(a,b)[0,1]
zot = np.roll(zot, max(range(-15,16), key=_c))
strokes = lambda s: int((((s-s.mean())[:-1]<0) & ((s-s.mean())[1:]>=0)).sum())
r = np.corrcoef(zmp[tv>3], zot[tv>3])[0,1]
plt.figure(figsize=(9, 3.4))
plt.plot(tv, zot, color="#0b7a52", lw=2, label="OptiTrack wrist (marker rig, 200 Hz)")
plt.plot(tv, zmp, color="#c026d3", lw=1.4, alpha=.85, label="MediaPipe wrist (single phone)")
plt.title(f"Same motion: {strokes(zmp)} strokes each  ·  r = {r:.2f} once rhythmic")
plt.xlabel("time (s)"); plt.ylabel("wrist position (wiping axis, norm.)")
plt.legend(); plt.grid(alpha=.2); plt.tight_layout(); plt.show()
print(f"MediaPipe vs OptiTrack:  {strokes(zmp)} vs {strokes(zot)} strokes,  r = {r:.2f} once rhythmic")

## Is it good enough? — the honest part

Single-camera markerless is **excellent** for **gross movement, timing, coordination, and counting** under good conditions — and it's **free, phone-portable, deployable anywhere**. It's **weaker** for absolute 3-D precision, contact/force, and fast or occluded motion.

The professional's question isn't *"is it as good as the rig?"* — it's **"is it good enough for _my_ application?"** Ask:
1. **What must I measure, how precisely?** angles / timing / counts → yes; mm-level 3-D → often no.
2. **My conditions?** good light, unobstructed → thrives; occlusion / speed → degrades.
3. **My accuracy budget?** would a 5° error change my decision? no → fine.
4. **Scale / cost / deployment?** field, many sites, no lab → markerless wins.
5. **Validate once, then trust:** check markerless vs a reference on a sample of your task, characterize the error, deploy within that budget — *exactly what the cell above did.*

**Discuss —** name a measurement problem from *your* work. Given what you just saw, is single-phone markerless good enough for it? What would you validate first?

## Design your own / your turn

- **Track a different joint** — change `joint`: knee (squats), shoulder (reaches), elbow (curls).
- **Count your reps** — the same peak-counter works on any cyclic movement; check it against your own count.
- **Compute speed** — `LM[:, 16, :2]` is the right-wrist path; the frame-to-frame distance is its speed. Where is it fastest?
- **Compare two clips** — record a "clean" and a "sloppy" rep; overlay the two angle curves.
- **On the day** — run this on **one of the two OpenCap phone videos**, then compare the single-phone MediaPipe read to the two-phone OpenCap 3-D of the *same* movement.

*Vibe-code freely — ask the assistant to modify any cell.*